# 📚 Módulo 02 - Monitoreo de Modelos en Producción

## 🎯 Objetivos

1. ✅ Entender por qué los modelos degradan en producción
2. ✅ Detectar data drift y concept drift
3. ✅ Monitorear performance en tiempo real
4. ✅ Implementar alertas automáticas

---

## 1️⃣ ¿Por qué Monitorear?

### Problema: Los Modelos se Degradan

```
Entrenamiento (t=0)     Producción (t=6 meses)
  Accuracy: 92%            Accuracy: 78%  ❌
  
  ¿Qué pasó?
```

**Causas comunes:**
* 📉 **Data drift**: Distribución de features cambió
* 🔄 **Concept drift**: Relación X → Y cambió
* 🐛 **Errores upstream**: ETL fallando
* 🌐 **Cambios externos**: Nueva regulación, competidores
* ⏱️ **Estacionalidad**: Patrones temporales no capturados

---

## 2️⃣ Data Drift (Cambio en Distribución)

### ¿Qué es Data Drift?

**Definición**: Cambio en la distribución de features de entrada

```python
# Entrenamiento
P_train(X) ≠ P_production(X)

Ejemplo:
  Train:      edad_promedio = 35 años
  Producción: edad_promedio = 52 años  ⚠️
```

### Tipos de Data Drift

#### 1. **Covariate Shift** (más común)
Cambia P(X), pero P(Y|X) constante

**Ejemplo**: Modelo de churn entrenado con datos de verano, usado en invierno
* Features cambian (temperatura, vacaciones)
* Relación churn → features igual

#### 2. **Prior Probability Shift**
Cambia P(Y), pero P(X|Y) constante

**Ejemplo**: Fraude aumenta de 1% a 5%
* Proporción de clases cambia
* Características del fraude igual

### Detección de Data Drift

**Métodos estadísticos:**

#### Kolmogorov-Smirnov (KS) Test
```python
from scipy.stats import ks_2samp

# Comparar distribución de una feature
statistic, p_value = ks_2samp(train_feature, prod_feature)

if p_value < 0.05:
    print("⚠️ Data drift detectado")
```

#### Population Stability Index (PSI)
```python
def calculate_psi(expected, actual, bins=10):
    """
    PSI mide divergencia entre dos distribuciones.
    PSI < 0.1: No drift
    0.1 < PSI < 0.2: Drift moderado
    PSI > 0.2: Drift significativo
    """
    expected_counts, bin_edges = np.histogram(expected, bins=bins)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)
    
    expected_pct = expected_counts / len(expected)
    actual_pct = actual_counts / len(actual)
    
    psi = np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct))
    return psi
```

#### Wasserstein Distance (Earth Mover's Distance)
```python
from scipy.stats import wasserstein_distance

dist = wasserstein_distance(train_feature, prod_feature)

if dist > threshold:
    print("⚠️ Distribución cambió significativamente")
```

---

## 3️⃣ Concept Drift (Cambio en Relación X → Y)

### ¿Qué es Concept Drift?

**Definición**: Cambio en la relación entre features y target

```python
# Antes
P(Y | X = x) = 0.8

# Después
P(Y | X = x) = 0.4  ⚠️
```

**Ejemplo**:
* **Antes**: "Clientes con > 5 compras no cancelan" (churn = 0)
* **Después**: Competidor ofrece mejor precio → clientes leales también se van

### Tipos de Concept Drift

#### 1. **Sudden Drift** (abrupto)
```
Performance
    ^
    |────────────┐
    |             │
    |             └───────>
    +────────────────> Time
              ^evento
```
**Causas**: Nueva regulación, cambio de producto

#### 2. **Gradual Drift** (progresivo)
```
Performance
    ^
    |────┐
    |     \
    |      \──────>
    +───────────────> Time
```
**Causas**: Cambios de mercado lentos

#### 3. **Incremental Drift**
```
Performance
    ^
    |─┐_┐_┐_┐
    |   \_\_\_\_\_\_>
    +───────────> Time
```
**Causas**: Muchos cambios pequeños

#### 4. **Recurring Drift** (cíclico)
```
Performance
    ^
    | /\  /\  /\
    |/  \/  \/  \
    +───────────> Time
```
**Causas**: Estacionalidad (Navidad, verano)

### Detección de Concept Drift

**Método principal**: Monitorear performance con ground truth

```python
# Requiere etiquetas verdaderas (a veces retrasadas)
from sklearn.metrics import accuracy_score

# Ventana deslizante de 7 días
window_accuracies = []
for day in range(len(predictions) - 7):
    window_pred = predictions[day:day+7]
    window_true = ground_truth[day:day+7]
    
    acc = accuracy_score(window_true, window_pred)
    window_accuracies.append(acc)

# Detectar caída significativa
baseline_acc = 0.85
if window_accuracies[-1] < baseline_acc - 0.05:
    print("⚠️ Concept drift detectado - Reentrenar modelo")
```

---

## 4️⃣ Métricas de Monitoreo

### Performance Metrics (con ground truth)

```python
# Clasificación
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score
)

metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred),
    "recall": recall_score(y_true, y_pred),
    "f1": f1_score(y_true, y_pred),
    "auc": roc_auc_score(y_true, y_proba)
}
```

**Regresión:**
```python
from sklearn.metrics import mean_absolute_error, mean_squared_error

metrics = {
    "mae": mean_absolute_error(y_true, y_pred),
    "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
    "mape": np.mean(np.abs((y_true - y_pred) / y_true)) * 100
}
```

### Proxy Metrics (sin ground truth inmediato)

```python
# Distribución de predicciones
predictions_dist = np.histogram(y_pred, bins=10)

# Confianza promedio
avg_confidence = np.mean(np.max(y_proba, axis=1))

# Predicciones constantes (red flag)
if len(np.unique(y_pred)) == 1:
    print("❌ Modelo predice siempre la misma clase")
```

### System Metrics

```python
# Latencia
import time

start = time.time()
prediction = model.predict(X)
latency_ms = (time.time() - start) * 1000

# Throughput
predictions_per_second = len(X) / (time.time() - start)

# Uso de memoria
import psutil
memory_usage_mb = psutil.Process().memory_info().rss / 1024 / 1024
```

---

## 5️⃣ Estrategias de Respuesta

### Cuando Detectas Drift

#### 1. **Reentrenar Modelo**
```python
if psi > 0.2 or accuracy < threshold:
    # Reentrenar con datos recientes
    new_model = train_model(recent_data)
    
    # Validar antes de deployment
    if validate(new_model) > current_performance:
        deploy(new_model)
```

#### 2. **Ajustar Threshold de Decisión**
```python
# Si prior P(Y) cambió, ajustar threshold
if fraud_rate_increased:
    # Reducir threshold para capturar más fraudes
    new_threshold = 0.3  # antes: 0.5
```

#### 3. **Feature Recalibration**
```python
# Actualizar estadísticas de normalización
scaler = StandardScaler()
scaler.fit(recent_data)  # en lugar de training_data
```

#### 4. **Ensemble con Modelo Nuevo**
```python
# Combinar modelo viejo y nuevo
prediction = 0.7 * old_model.predict(X) + 0.3 * new_model.predict(X)
```

---

## 6️⃣ Arquitectura de Monitoreo

```
┌───────────────────┐
│  Prediction Service  │
│  (Model Serving)     │
└─────────┬──────────┘
         │
         │ Log (input, output, metadata)
         ↓
┌─────────┬───────────────┐
│  Delta Lake (Prediction Logs) │
└─────────┬───────────────┘
         │
         │ Streaming job (cada hora)
         ↓
┌─────────┬───────────────┐
│  Drift Detection Job      │
│  - PSI, KS test           │
│  - Performance metrics    │
└─────────┬───────────────┘
         │
         │ Alertas
         ↓
┌─────────┬───────────────┐
│  Dashboard + Slack/Email  │
└─────────────────────────┘
```

---

## ✅ Mejores Prácticas

1. **Log everything**: Inputs, outputs, timestamps, metadata
2. **Ventanas móviles**: Comparar con periodo similar (mismo día de semana)
3. **Múltiples detectores**: No confiar en una sola métrica
4. **Ground truth retrasado**: Planear para etiquetas tardías
5. **Automatic retraining**: Con validación antes de deployment
6. **A/B testing**: Probar modelo nuevo en % pequeño de tráfico
7. **Rollback rápido**: Mantener versión anterior lista

---

**Universidad del Aconcagua 🇦🇷**